In [1]:
import os
import ast
import numpy as np
import xarray as xr
from workers.processing import find_centroid, find_stars
from workers.compression import crop_centre, crop_sidelobes

import matplotlib.pyplot as plt

In [2]:
image_dir = 'images/simulated/'
image_paths = sorted([
            os.path.join(image_dir, f)
            for f in os.listdir(image_dir)
            if f.lower().endswith('.npy')
        ])

In [4]:
images_grouped = [image_paths[i:i+10] for i in range(0, len(image_paths), 10)]

In [5]:
for chunk in images_grouped:
    print(chunk)

['images/simulated/toliman_image_00.npy', 'images/simulated/toliman_image_01.npy', 'images/simulated/toliman_image_02.npy', 'images/simulated/toliman_image_03.npy', 'images/simulated/toliman_image_04.npy', 'images/simulated/toliman_image_05.npy', 'images/simulated/toliman_image_06.npy', 'images/simulated/toliman_image_07.npy', 'images/simulated/toliman_image_08.npy', 'images/simulated/toliman_image_09.npy']
['images/simulated/toliman_image_10.npy', 'images/simulated/toliman_image_11.npy', 'images/simulated/toliman_image_12.npy', 'images/simulated/toliman_image_13.npy', 'images/simulated/toliman_image_14.npy', 'images/simulated/toliman_image_15.npy', 'images/simulated/toliman_image_16.npy', 'images/simulated/toliman_image_17.npy', 'images/simulated/toliman_image_18.npy', 'images/simulated/toliman_image_19.npy']


In [6]:
chunk

['images/simulated/toliman_image_10.npy',
 'images/simulated/toliman_image_11.npy',
 'images/simulated/toliman_image_12.npy',
 'images/simulated/toliman_image_13.npy',
 'images/simulated/toliman_image_14.npy',
 'images/simulated/toliman_image_15.npy',
 'images/simulated/toliman_image_16.npy',
 'images/simulated/toliman_image_17.npy',
 'images/simulated/toliman_image_18.npy',
 'images/simulated/toliman_image_19.npy']

In [59]:
def crop_image_with_metadata(raw_filename):

    # Load raw image
    raw_image = np.load(raw_filename)

    # Load raw image metadata
    metadata_file = raw_filename.strip(".npy")+".txt"

    with open(metadata_file, "r") as file:
        for line in file:
            header = ast.literal_eval(line.strip())

    # Crop core and sidelobes from raw image
    centroid_data = find_centroid(raw_image)
    core = crop_centre(raw_image, centroid_data["x"], centroid_data["y"])
    star_poss = find_stars(core)
    x_poss = np.round(star_poss['xs'] + centroid_data['x'] - core.shape[1]//2)
    y_poss = np.round(star_poss['ys'] + centroid_data['y'] - core.shape[0]//2)
    sidelobes = crop_sidelobes(raw_image, x_poss, y_poss, centroid_data)

    # Convert CAMTIME to a string to avoid truncation/error when creating netCDF file
    header["CAMTIME"] = str(header["CAMTIME"])

    # Add position information to header
    header["CENTR_X"] = int(np.round(centroid_data['x']))
    header["CENTR_Y"] = int(np.round(centroid_data['y']))
    header["STAR_1_X"] = int(x_poss[0])
    header["STAR_1_Y"] = int(y_poss[0])
    header["STAR_2_X"] = int(x_poss[1])
    header["STAR_2_Y"] = int(y_poss[1])

    return core.astype(np.int16), sidelobes.astype(np.int16), header

In [60]:
# Get reference image
ref_core, ref_sidelobes, header = crop_image_with_metadata(chunk[0])

diff_cores = []
diff_sidelobes = []
header = [header]

# Get differences from reference for remaining images
for filename in chunk[1:]:

    core, sidelobes, metadata = crop_image_with_metadata(filename)

    diff_cores.append(core - ref_core)
    diff_sidelobes.append(sidelobes - ref_sidelobes)
    header.append(metadata)

diff_cores = np.asarray(diff_cores)
diff_sidelobes = np.asarray(diff_sidelobes)

# Convert to int8 if safe to do so
if np.max(np.abs(diff_cores)) <= 127:
    diff_cores = diff_cores.astype(np.int8)

if np.max(np.abs(diff_sidelobes)) <= 127:
    diff_sidelobes = diff_sidelobes.astype(np.int8)

# Create xarray Dataset
ref_core_array = xr.DataArray(ref_core, dims=("y", "x"))
diff_core_array = xr.DataArray(diff_cores, dims=("i", "y", "x"))
ref_side_array = xr.DataArray(ref_sidelobes, dims=("a","b"))
diff_side_array = xr.DataArray(diff_sidelobes, dims=("i", "a", "b"))

dataset = xr.Dataset({
    "ref_core": ref_core_array,
    "ref_sidelobes": ref_side_array,
    "diff_core": diff_core_array,
    "diff_sidelobes": diff_side_array
})

In [68]:
header

[{'SEQNUM': 12,
  'CAMTIME': '22541562552104',
  'COMTIME': '2025-04-08 15:55:45.020529',
  'EXPOSURE': 46.0,
  'PXLFMT': 'Mono16',
  'XOFF': 912,
  'YOFF': 0,
  'XPAD': 0,
  'YPAD': 0,
  'CENTR_X': 1823,
  'CENTR_Y': 1824,
  'STAR_1_X': 1829,
  'STAR_1_Y': 1833,
  'STAR_2_X': 1819,
  'STAR_2_Y': 1815},
 {'SEQNUM': 13,
  'CAMTIME': '22541662554608',
  'COMTIME': '2025-04-08 15:55:45.122598',
  'EXPOSURE': 46.0,
  'PXLFMT': 'Mono16',
  'XOFF': 912,
  'YOFF': 0,
  'XPAD': 0,
  'YPAD': 0,
  'CENTR_X': 1824,
  'CENTR_Y': 1824,
  'STAR_1_X': 1829,
  'STAR_1_Y': 1833,
  'STAR_2_X': 1819,
  'STAR_2_Y': 1815},
 {'SEQNUM': 14,
  'CAMTIME': '22541762557424',
  'COMTIME': '2025-04-08 15:55:45.220023',
  'EXPOSURE': 46.0,
  'PXLFMT': 'Mono16',
  'XOFF': 912,
  'YOFF': 0,
  'XPAD': 0,
  'YPAD': 0,
  'CENTR_X': 1824,
  'CENTR_Y': 1824,
  'STAR_1_X': 1829,
  'STAR_1_Y': 1833,
  'STAR_2_X': 1819,
  'STAR_2_Y': 1815},
 {'SEQNUM': 15,
  'CAMTIME': '22541862559104',
  'COMTIME': '2025-04-08 15:55:45.3276

In [ ]:
for i, d in enumerate(header):
    h["i"+


In [62]:
h = {i: d for i, d in enumerate(header)}

In [73]:
x = {str(i)+'_'+k: v for i, d in enumerate(header) for k, v in d.items()}

In [74]:
x

{'0_SEQNUM': 12,
 '0_CAMTIME': '22541562552104',
 '0_COMTIME': '2025-04-08 15:55:45.020529',
 '0_EXPOSURE': 46.0,
 '0_PXLFMT': 'Mono16',
 '0_XOFF': 912,
 '0_YOFF': 0,
 '0_XPAD': 0,
 '0_YPAD': 0,
 '0_CENTR_X': 1823,
 '0_CENTR_Y': 1824,
 '0_STAR_1_X': 1829,
 '0_STAR_1_Y': 1833,
 '0_STAR_2_X': 1819,
 '0_STAR_2_Y': 1815,
 '1_SEQNUM': 13,
 '1_CAMTIME': '22541662554608',
 '1_COMTIME': '2025-04-08 15:55:45.122598',
 '1_EXPOSURE': 46.0,
 '1_PXLFMT': 'Mono16',
 '1_XOFF': 912,
 '1_YOFF': 0,
 '1_XPAD': 0,
 '1_YPAD': 0,
 '1_CENTR_X': 1824,
 '1_CENTR_Y': 1824,
 '1_STAR_1_X': 1829,
 '1_STAR_1_Y': 1833,
 '1_STAR_2_X': 1819,
 '1_STAR_2_Y': 1815,
 '2_SEQNUM': 14,
 '2_CAMTIME': '22541762557424',
 '2_COMTIME': '2025-04-08 15:55:45.220023',
 '2_EXPOSURE': 46.0,
 '2_PXLFMT': 'Mono16',
 '2_XOFF': 912,
 '2_YOFF': 0,
 '2_XPAD': 0,
 '2_YPAD': 0,
 '2_CENTR_X': 1824,
 '2_CENTR_Y': 1824,
 '2_STAR_1_X': 1829,
 '2_STAR_1_Y': 1833,
 '2_STAR_2_X': 1819,
 '2_STAR_2_Y': 1815,
 '3_SEQNUM': 15,
 '3_CAMTIME': '22541862559

In [63]:
h

{0: {'SEQNUM': 12,
  'CAMTIME': '22541562552104',
  'COMTIME': '2025-04-08 15:55:45.020529',
  'EXPOSURE': 46.0,
  'PXLFMT': 'Mono16',
  'XOFF': 912,
  'YOFF': 0,
  'XPAD': 0,
  'YPAD': 0,
  'CENTR_X': 1823,
  'CENTR_Y': 1824,
  'STAR_1_X': 1829,
  'STAR_1_Y': 1833,
  'STAR_2_X': 1819,
  'STAR_2_Y': 1815},
 1: {'SEQNUM': 13,
  'CAMTIME': '22541662554608',
  'COMTIME': '2025-04-08 15:55:45.122598',
  'EXPOSURE': 46.0,
  'PXLFMT': 'Mono16',
  'XOFF': 912,
  'YOFF': 0,
  'XPAD': 0,
  'YPAD': 0,
  'CENTR_X': 1824,
  'CENTR_Y': 1824,
  'STAR_1_X': 1829,
  'STAR_1_Y': 1833,
  'STAR_2_X': 1819,
  'STAR_2_Y': 1815},
 2: {'SEQNUM': 14,
  'CAMTIME': '22541762557424',
  'COMTIME': '2025-04-08 15:55:45.220023',
  'EXPOSURE': 46.0,
  'PXLFMT': 'Mono16',
  'XOFF': 912,
  'YOFF': 0,
  'XPAD': 0,
  'YPAD': 0,
  'CENTR_X': 1824,
  'CENTR_Y': 1824,
  'STAR_1_X': 1829,
  'STAR_1_Y': 1833,
  'STAR_2_X': 1819,
  'STAR_2_Y': 1815},
 3: {'SEQNUM': 15,
  'CAMTIME': '22541862559104',
  'COMTIME': '2025-04-08 1

In [67]:
h[0]['CAMTIME']

'22541562552104'

In [7]:
raw_filename = chunk[0]

# Load raw image
raw_image = np.load(raw_filename)

# Load raw image metadata
metadata_file = raw_filename.strip(".npy")+".txt"

with open(metadata_file, "r") as file:
    for line in file:
        header = ast.literal_eval(line.strip())

# Crop core and sidelobes from raw image
centroid_data = find_centroid(raw_image)
core = crop_centre(raw_image, centroid_data["x"], centroid_data["y"])
star_poss = find_stars(core)
x_poss = np.round(star_poss['xs'] + centroid_data['x'] - core.shape[1]//2)
y_poss = np.round(star_poss['ys'] + centroid_data['y'] - core.shape[0]//2)
# sidelobes = crop_sidelobes(raw_image, x_poss, y_poss)
sidelobes = crop_sidelobes(raw_image, x_poss, y_poss, centroid_data)

# Create xarray Dataset
core_array = xr.DataArray(core, dims=("y", "x"))
sidelobe_array = xr.DataArray(sidelobes)

dataset = xr.Dataset({
    "core": core_array,
    "sidelobes": sidelobe_array
})

# Convert CAMTIME to a string to avoid truncation/error when creating netCDF file
header["CAMTIME"] = str(header["CAMTIME"])

# Add position information to header
header["CENTR_X"] = np.round(centroid_data['x'])
header["CENTR_Y"] = np.round(centroid_data['y'])
header["STAR_1_X"] = x_poss[0]
header["STAR_1_Y"] = y_poss[0]
header["STAR_2_X"] = x_poss[1]
header["STAR_2_Y"] = y_poss[1]

dataset.attrs = header

# Set compression encoding
dataset["core"].encoding = {"zlib": True, "complevel": 9}
dataset["sidelobes"].encoding = {"zlib": True, "complevel": 9}

In [8]:
ref_core = core
ref_sidelobes = sidelobes

In [43]:
diff_cores = []
diff_sidelobes = []

for raw_filename in chunk[1:]:
    # Load raw image
    raw_image = np.load(raw_filename)
        
    # Load raw image metadata
    metadata_file = raw_filename.strip(".npy")+".txt"
    
    with open(metadata_file, "r") as file:
        for line in file:
            header = ast.literal_eval(line.strip())
    
    # Crop core and sidelobes from raw image
    centroid_data = find_centroid(raw_image)
    core = crop_centre(raw_image, centroid_data["x"], centroid_data["y"])
    star_poss = find_stars(core)
    x_poss = np.round(star_poss['xs'] + centroid_data['x'] - core.shape[1]//2)
    y_poss = np.round(star_poss['ys'] + centroid_data['y'] - core.shape[0]//2)
    # sidelobes = crop_sidelobes(raw_image, x_poss, y_poss)
    sidelobes = crop_sidelobes(raw_image, x_poss, y_poss, centroid_data)
    
    diff_cores.append(core.astype(np.int16) - ref_core.astype(np.int16))
    diff_sidelobes.append(sidelobes.astype(np.int16) - ref_sidelobes.astype(np.int16))

In [44]:
diff_cores = np.asarray(diff_cores)
diff_sidelobes = np.asarray(diff_sidelobes)

In [45]:
print(diff_cores.nbytes)
print(diff_sidelobes.nbytes)


294912
311040


In [46]:
if np.max(np.abs(diff_cores)) <= 127:
    diff_cores = diff_cores.astype(np.int8)

if np.max(np.abs(diff_sidelobes)) <= 127:
    diff_sidelobes = diff_sidelobes.astype(np.int8)

In [47]:
print(diff_cores.nbytes + diff_sidelobes.nbytes)

450432


In [48]:
9*dataset.nbytes

605952

In [51]:
ref_core_array = xr.DataArray(ref_core, dims=("y", "x"))
diff_core_array = xr.DataArray(diff_cores, dims=("i", "y", "x"))
ref_side_array = xr.DataArray(ref_sidelobes, dims=("a","b"))
diff_side_array = xr.DataArray(diff_sidelobes, dims=("i", "a", "b"))

In [52]:
new_dataset = xr.Dataset({
    "ref_core": ref_core_array,
    "ref_sidelobes": ref_side_array,
    "diff_core": diff_core_array,
    "diff_sidelobes": diff_side_array
})

In [53]:
new_dataset.nbytes

517760

In [54]:
10*dataset.nbytes

673280

In [10]:
np.max(core.astype(np.int16) - ref_core.astype(np.int16))

np.int16(26)

In [12]:
np.min(core.astype(np.int16) - ref_core.astype(np.int16))

np.int16(-627)

In [54]:
np.max(sidelobes.astype(np.int16) - ref_sideloebs.astype(np.int16))

np.int16(13)

In [23]:
dataset

<xarray.Dataset> Size: 67kB
Dimensions:    (y: 128, x: 128, dim_0: 2880, dim_1: 6)
Dimensions without coordinates: y, x, dim_0, dim_1
Data variables:
    core       (y, x) uint16 33kB 2 2 2 2 2 1 2 1 2 2 3 ... 2 2 2 2 3 3 2 2 2 2
    sidelobes  (dim_0, dim_1) uint16 35kB 1 1 1 2 1 2 1 2 1 ... 1 1 1 1 2 1 1 2
Attributes: (12/15)
    SEQNUM:    12
    CAMTIME:   22541562552104
    COMTIME:   2025-04-08 15:55:45.020529
    EXPOSURE:  46.0
    PXLFMT:    Mono16
    XOFF:      912
    ...        ...
    CENTR_X:   1823.0
    CENTR_Y:   1824.0
    STAR_1_X:  1829.0
    STAR_1_Y:  1833.0
    STAR_2_X:  1819.0
    STAR_2_Y:  1815.0

In [28]:
np.iinfo(np.uint16)

iinfo(min=0, max=65535, dtype=uint16)

In [32]:
np.iinfo(np.int8)

iinfo(min=-128, max=127, dtype=int8)

In [33]:
np.iinfo(np.uint8)

iinfo(min=0, max=255, dtype=uint8)

In [27]:
2**8

256

In [25]:
np.max(core)

np.uint16(642)

In [26]:
np.max(sidelobes)

np.uint16(19)

In [36]:
np.min(core.astype(np.int8))

np.int8(-128)

In [75]:
data = xr.open_dataset("images/compressed/frame_proc_1744256617875346.nc")

In [79]:
from datetime import datetime

In [80]:
camtimes = []
comtimes = []
for k, v in data.attrs.items():
    if 'CAMTIME' in k:
        camtimes.append(int(v))
    if 'COMTIME' in k:
        comtimes.append(datetime.strptime(v, '%Y-%m-%d %H:%M:%S.%f'))